# 04. Robust Multi-model Ensemble

`03_feature_engineering`의 모델 다양성을 유지하면서 개별 모델의 분산을 줄이는 Train-only 앙상블입니다. 개발 과정에서 ㎡당 가격 모델, 과거 집계 피처, 자동 스태킹과 세분화된 상호작용 타깃 인코딩을 시간 Fold에서 비교했으나 개선이 일관되지 않아 제외했습니다.

최종 구성은 다음과 같습니다.

- Leave-one-out Target Encoding + 원가격 Linear Tree LightGBM 3개 복잡도 평균
- 로그 Target Linear Tree LightGBM
- CatBoost 4개 random seed 평균
- 상위 모델 가중치 `0.55 / 0.30 / 0.15`는 03에서 결정한 값을 고정하여 재튜닝하지 않음

모든 전처리와 Target 통계는 각 Fold의 과거 Train에만 fit합니다. Test는 마지막 단계에서 `transform`과 `predict`에만 사용합니다.

In [1]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 70)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
RANDOM_STATE = 42

In [2]:
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'seoul_real_estate_train.csv')
test = pd.read_csv(DATA_DIR / 'seoul_real_estate_test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Test의 값이나 분포는 출력하거나 모델 선택에 사용하지 않습니다.
print('Train:', train.shape)
print('Train period:', train['Transaction_YearMonth'].min(), '~', train['Transaction_YearMonth'].max())

Train: (1969, 11)
Train period: 202401 ~ 202512


In [3]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def rmsle(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return rmse(np.log1p(y_true), np.log1p(y_pred))

CATEGORICAL_COLS = ['Gu', 'Dong', 'Gu_Dong']
TE_LEAVES = [3, 4, 7]
CAT_SEEDS = [21, 42, 84, 2026]
BLEND_WEIGHTS = {'TE_LGBM_Ensemble': 0.55, 'Log_LGBM': 0.30, 'CatBoost_Ensemble': 0.15}
assert np.isclose(sum(BLEND_WEIGHTS.values()), 1.0)

## 1. 공통 피처 엔지니어링

In [4]:
def engineer_features(df, subway_median):
    x = df.copy().reset_index(drop=True)
    x['Transaction_Year'] = x['Transaction_YearMonth'] // 100
    x['Transaction_Month'] = x['Transaction_YearMonth'] % 100
    x['Time_Index'] = (x['Transaction_Year'] - 2024) * 12 + x['Transaction_Month'] - 1
    x['Age_at_Transaction'] = x['Transaction_Year'] - x['Year_Built']
    x['Gu_Dong'] = x['Gu'].astype(str) + '_' + x['Dong'].astype(str)

    x['Subway_Distance_Missing'] = x['Distance_to_Subway'].isna().astype('int8')
    x['Distance_to_Subway'] = x['Distance_to_Subway'].fillna(subway_median)
    x['Log_Area'] = np.log(x['Exclusive_Area'])
    x['Area_Squared'] = x['Exclusive_Area'] ** 2
    x['Area_Cubed_Scaled'] = x['Exclusive_Area'] ** 3 / 10_000
    x['Floor_Squared'] = x['Floor'] ** 2
    x['Distance_Squared'] = x['Distance_to_Subway'] ** 2
    x['Age_Squared'] = x['Age_at_Transaction'] ** 2
    x['Time_Squared'] = x['Time_Index'] ** 2

    return x.drop(columns=['ID', 'Target', 'Transaction_YearMonth', 'Year_Built'], errors='ignore')

def fit_one_hot_transformer(fit_df):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    numeric_cols = [c for c in fit_raw.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    encoder.fit(fit_raw[CATEGORICAL_COLS])

    def transform(df):
        raw = engineer_features(df, subway_median)
        numeric = raw[numeric_cols].reset_index(drop=True)
        encoded = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, encoded], axis=1)

    return transform, subway_median

## 2. 완전 Leave-one-out Target Encoding

In [5]:
def add_strict_loo_encodings(fit_df, other_df, fit_raw, other_raw, prior=5.0):
    fit_reset = fit_df.reset_index(drop=True)
    other_reset = other_df.reset_index(drop=True)
    fit_encoded = fit_raw.copy()
    other_encoded = other_raw.copy()

    targets = {
        'Target_Mean': fit_reset['Target'],
        'Price_Per_Area_Mean': fit_reset['Target'] / fit_reset['Exclusive_Area'],
    }
    for suffix, values in targets.items():
        global_sum = values.sum()
        global_count = len(values)
        global_mean = global_sum / global_count
        loo_global_mean = (global_sum - values) / (global_count - 1)

        for group_col in ['Gu', 'Dong']:
            source = pd.DataFrame({group_col: fit_reset[group_col], 'value': values})
            group_sum = source.groupby(group_col)['value'].transform('sum')
            group_count = source.groupby(group_col)['value'].transform('count')
            fit_encoded[f'{group_col}_{suffix}'] = (
                group_sum - values + prior * loo_global_mean
            ) / (group_count - 1 + prior)

            full_stats = source.groupby(group_col)['value'].agg(['sum', 'count'])
            mapping = (full_stats['sum'] + prior * global_mean) / (full_stats['count'] + prior)
            other_encoded[f'{group_col}_{suffix}'] = (
                other_reset[group_col].map(mapping).fillna(global_mean).to_numpy()
            )
    return fit_encoded, other_encoded

def fit_target_encoded_pair(fit_df, other_df, prior=5.0):
    subway_median = fit_df['Distance_to_Subway'].median()
    fit_raw = engineer_features(fit_df, subway_median)
    other_raw = engineer_features(other_df, subway_median)
    fit_te, other_te = add_strict_loo_encodings(
        fit_df, other_df, fit_raw, other_raw, prior=prior
    )
    numeric_cols = [c for c in fit_te.columns if c not in CATEGORICAL_COLS]
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    encoder.fit(fit_te[CATEGORICAL_COLS])

    def encode(raw):
        numeric = raw[numeric_cols].reset_index(drop=True)
        categorical = pd.DataFrame(
            encoder.transform(raw[CATEGORICAL_COLS]),
            columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
        )
        return pd.concat([numeric, categorical], axis=1)

    return encode(fit_te), encode(other_te)

## 3. 모델 설정

In [6]:
COMMON_LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'n_estimators': 4000, 'learning_rate': 0.015,
    'min_child_samples': 40, 'feature_fraction': 1.0,
    'bagging_fraction': 1.0, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'linear_tree': True, 'linear_lambda': 10.0,
    'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 1,
}
CAT_PARAMS = {
    'iterations': 2500, 'learning_rate': 0.03, 'depth': 5,
    'l2_leaf_reg': 3.0, 'loss_function': 'RMSE', 'eval_metric': 'RMSE',
    'random_strength': 0.5, 'bootstrap_type': 'Bayesian',
    'bagging_temperature': 0.5, 'verbose': False,
    'allow_writing_files': False, 'thread_count': 1,
}

## 4. Train-only 시간 기반 다중 검증

Fold는 Test를 조회하지 않고 Train의 마지막 9개 거래월만으로 자동 생성합니다. 각 검증 Fold보다 과거인 데이터만 학습합니다.

In [7]:
train_months = np.sort(train['Transaction_YearMonth'].unique())
validation_month_groups = np.array_split(train_months[-9:], 3)
TIME_FOLDS = [
    (f'Train-derived Fold {i}', months)
    for i, months in enumerate(validation_month_groups, start=1)
]

fold_results = []
iteration_history = {
    'TE_LGBM': {leaves: [] for leaves in TE_LEAVES},
    'Log_LGBM': [],
    'CatBoost': {seed: [] for seed in CAT_SEEDS},
}
oof_true, oof_03, oof_04 = [], [], []

for fold_name, valid_months in TIME_FOLDS:
    fit_df = train.loc[train['Transaction_YearMonth'] < valid_months.min()].copy()
    valid_df = train.loc[train['Transaction_YearMonth'].isin(valid_months)].copy()
    assert fit_df['Transaction_YearMonth'].max() < valid_df['Transaction_YearMonth'].min()

    # Target-encoded LightGBM complexity ensemble
    X_te_fit, X_te_valid = fit_target_encoded_pair(fit_df, valid_df)
    te_predictions = []
    for leaves in TE_LEAVES:
        model = lgb.LGBMRegressor(**COMMON_LGB_PARAMS, num_leaves=leaves)
        model.fit(
            X_te_fit, fit_df['Target'].reset_index(drop=True),
            eval_set=[(X_te_valid, valid_df['Target'].reset_index(drop=True))],
            callbacks=[lgb.early_stopping(150, verbose=False)],
        )
        te_predictions.append(model.predict(X_te_valid))
        iteration_history['TE_LGBM'][leaves].append(model.best_iteration_)
    pred_te_single = te_predictions[0]
    pred_te_ensemble = np.mean(te_predictions, axis=0)

    # Log-target LightGBM
    transform, subway_median = fit_one_hot_transformer(fit_df)
    X_fit, X_valid = transform(fit_df), transform(valid_df)
    log_model = lgb.LGBMRegressor(**COMMON_LGB_PARAMS, num_leaves=4)
    log_model.fit(
        X_fit, np.log1p(fit_df['Target']).reset_index(drop=True),
        eval_set=[(X_valid, np.log1p(valid_df['Target']).reset_index(drop=True))],
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )
    pred_log = np.expm1(log_model.predict(X_valid))
    iteration_history['Log_LGBM'].append(log_model.best_iteration_)

    # CatBoost seed ensemble
    cat_fit = engineer_features(fit_df, subway_median)
    cat_valid = engineer_features(valid_df, subway_median)
    cat_predictions = []
    for seed in CAT_SEEDS:
        cat_model = CatBoostRegressor(**CAT_PARAMS, random_seed=seed)
        cat_model.fit(
            cat_fit, fit_df['Target'].reset_index(drop=True),
            cat_features=CATEGORICAL_COLS,
            eval_set=(cat_valid, valid_df['Target'].reset_index(drop=True)),
            early_stopping_rounds=150, verbose=False,
        )
        cat_predictions.append(cat_model.predict(cat_valid))
        iteration_history['CatBoost'][seed].append(cat_model.get_best_iteration() + 1)
    pred_cat_single = cat_predictions[CAT_SEEDS.index(42)]
    pred_cat_ensemble = np.mean(cat_predictions, axis=0)

    # 03 재현과 04 robust ensemble 비교: 상위 가중치는 고정
    pred_03 = 0.55 * pred_te_single + 0.30 * pred_log + 0.15 * pred_cat_single
    pred_04 = 0.55 * pred_te_ensemble + 0.30 * pred_log + 0.15 * pred_cat_ensemble
    y_valid = valid_df['Target'].to_numpy()
    fold_results.append({
        'Fold': fold_name, 'Train_rows': len(fit_df), 'Valid_rows': len(valid_df),
        '03_RMSE': rmse(y_valid, pred_03),
        '04_RMSE': rmse(y_valid, pred_04),
        '04_RMSLE': rmsle(y_valid, pred_04),
        'Improvement': rmse(y_valid, pred_03) - rmse(y_valid, pred_04),
    })
    oof_true.extend(y_valid); oof_03.extend(pred_03); oof_04.extend(pred_04)

scores = pd.DataFrame(fold_results)
display(scores)

,Fold,Train_rows,Valid_rows,03_RMSE,04_RMSE,04_RMSLE,Improvement
0,Train-derived Fold 1,1201,278,"2,100.895856","2,107.743326",0.055311,-6.847470
1,Train-derived Fold 2,1479,244,"2,644.800383","2,634.763161",0.080742,10.037222
2,Train-derived Fold 3,1723,246,"2,536.907840","2,525.601387",0.071630,11.306454


In [8]:
pooled_03 = rmse(oof_true, oof_03)
pooled_04 = rmse(oof_true, oof_04)
improved_folds = int((scores['04_RMSE'] < scores['03_RMSE']).sum())
print(f'03 pooled RMSE: {pooled_03:,.2f}')
print(f'04 pooled RMSE: {pooled_04:,.2f}')
print(f'Improved folds : {improved_folds}/{len(scores)}')
print(f'Pooled improvement: {(pooled_03 - pooled_04) / pooled_03:.2%}')
assert pooled_04 < pooled_03
assert improved_folds >= 2

03 pooled RMSE: 2,425.19
04 pooled RMSE: 2,420.08
Improved folds : 2/3
Pooled improvement: 0.21%


## 5. 전체 Train 재학습 및 최종 예측

In [9]:
# Test는 이 셀에서 처음으로 피처 transform과 predict에만 사용됩니다.
X_te_full, X_te_test = fit_target_encoded_pair(train, test)
te_test_predictions = []
for leaves in TE_LEAVES:
    rounds = int(np.median(iteration_history['TE_LGBM'][leaves]))
    model = lgb.LGBMRegressor(
        **{**COMMON_LGB_PARAMS, 'num_leaves': leaves, 'n_estimators': rounds}
    )
    model.fit(X_te_full, train['Target'].reset_index(drop=True))
    te_test_predictions.append(model.predict(X_te_test))
test_pred_te = np.mean(te_test_predictions, axis=0)

final_transform, full_median = fit_one_hot_transformer(train)
X_full, X_test = final_transform(train), final_transform(test)
log_rounds = int(np.median(iteration_history['Log_LGBM']))
log_model = lgb.LGBMRegressor(
    **{**COMMON_LGB_PARAMS, 'num_leaves': 4, 'n_estimators': log_rounds}
)
log_model.fit(X_full, np.log1p(train['Target']).reset_index(drop=True))
test_pred_log = np.expm1(log_model.predict(X_test))

cat_full = engineer_features(train, full_median)
cat_test = engineer_features(test, full_median)
cat_test_predictions = []
for seed in CAT_SEEDS:
    rounds = int(np.median(iteration_history['CatBoost'][seed]))
    model = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed, 'iterations': rounds})
    model.fit(
        cat_full, train['Target'].reset_index(drop=True),
        cat_features=CATEGORICAL_COLS, verbose=False,
    )
    cat_test_predictions.append(model.predict(cat_test))
test_pred_cat = np.mean(cat_test_predictions, axis=0)

test_pred = np.clip(
    BLEND_WEIGHTS['TE_LGBM_Ensemble'] * test_pred_te
    + BLEND_WEIGHTS['Log_LGBM'] * test_pred_log
    + BLEND_WEIGHTS['CatBoost_Ensemble'] * test_pred_cat,
    0, None,
)

## 6. Leakage Audit 및 제출 파일 생성

In [10]:
leakage_audit = pd.Series({
    'Imputation statistics fitted on Train only': True,
    'OneHotEncoder fitted on Train only': True,
    'No independent dummy encoding on Test': True,
    'Target encoding excludes each training row target': True,
    'Validation uses strictly earlier transactions': True,
    'Weights and rounds selected without Test': True,
    'Test limited to transform and predict': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred})
submission = sample_submission[['ID']].merge(
    prediction_frame, on='ID', how='left', validate='one_to_one'
)
assert submission['Target'].notna().all()
assert len(submission) == len(sample_submission)

output_path = SUBMISSION_DIR / '04_robust_ensemble_submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
display(submission.head())
display(submission['Target'].describe())

Imputation statistics fitted on Train only           True
OneHotEncoder fitted on Train only                   True
No independent dummy encoding on Test                True
Target encoding excludes each training row target    True
Validation uses strictly earlier transactions        True
Weights and rounds selected without Test             True
Test limited to transform and predict                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/04_robust_ensemble_submission.csv


,ID,Target
0,TR_2427,"36,507.968817"
1,TR_0028,"48,207.770746"
2,TR_1461,"29,105.254981"
3,TR_1977,"47,062.173095"
4,TR_2351,"47,245.294134"


count      531.000000
mean    39,919.922321
std      9,656.266082
min     13,957.682730
25%     32,933.668400
50%     39,556.488213
75%     47,093.393519
max     64,815.616234
Name: Target, dtype: float64

## 실험 결론

04는 새로운 Test 정보나 Public Score로 가중치를 조정하지 않습니다. 03과 동일한 상위 가중치를 유지하면서, 원가격 LightGBM의 트리 복잡도와 CatBoost의 seed에 따른 분산만 평균합니다. 시간 검증에서 pooled RMSE가 낮고 세 Fold 중 최소 두 Fold가 개선될 때만 제출 파일을 생성합니다.